## Warehouse Location and Supply Chain

A company wants to open new warehouses to supply products to a set of 10 retail stores in a region. The goal is to minimize the total cost of opening warehouses and transporting goods to the stores.

- There are 20 potential sites for the warehouses.
- Each potential warehouse location has a fixed cost associated with opening it.
- Each retail store has a given demand that must be satisfied.
- The cost of transporting goods from each warehouse to each store is known.
- A warehouse, once opened, can supply goods to multiple stores, but each store must be supplied by exactly one warehouse.
- There is a constraint on the maximum number of warehouses that can be opened due to budget limitations.
- Each warehouse can send a maximum number of units.

Describe the parameters and variables of the problem and write the mathematical model.


**INDEXES**:
- $i \in I$ : potential warehouse locations ($|I| = 20$)
- $j \in J$ : retail stores ($|J| = 10$)

**PARAMETERS**:
- $f_i$ : fixed cost of opening warehouse $i$
- $d_j$ : demand of retail store $j$
- $c_{i,j}$ : unit transportation cost from warehouse $i$ to store $j$
- $K$ : maximum number of warehouses that can be opened
- $U$ : maximum number of units each warehouse can send

**VARIABLES**:
- $x_i \in \{0,1\}$ : 1 if warehouse $i$ is opened, 0 otherwise
- $y_{i,j} \in \{0,1\}$ : 1 if store $j$ is supplied by warehouse $i$
- $z_{i,j} \ge 0$ : amount of units shipped from warehouse $i$ to store $j$

$$ \min \ \ \sum_{i \in I} f_i \cdot x_i + \sum_{i \in I} \sum_{j \in J} c_{i,j} \cdot z_{i,j} $$

s.t.

$$ \sum_{i \in I} y_{i,j} = 1 \quad \forall j \in J \quad \text{(each store supplied by exactly one warehouse)} $$

$$ y_{i,j} \le x_i \quad \forall i \in I, \forall j \in J \quad \text{(can only supply from an open warehouse)} $$

$$ z_{i,j} = d_j \cdot y_{i,j} \quad \forall i \in I, \forall j \in J \quad \text{(ship exactly the demand if assigned)} $$

$$ \sum_{j \in J} z_{i,j} \le U \cdot x_i \quad \forall i \in I \quad \text{(warehouse capacity)} $$

$$ \sum_{i \in I} x_i \le K \quad \text{(budget limit on number of warehouses)} $$

$$ x_i \in \{0,1\}, \ y_{i,j} \in \{0,1\}, \ z_{i,j} \ge 0 $$

In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

np.random.seed(42)
m = gp.Model("supplychain")

num_warehouses = 20
num_stores = 10
I = list(range(num_warehouses))
J = list(range(num_stores))

fixed_costs = np.random.uniform(50000, 100000, num_warehouses)
demands = np.random.randint(100, 500, num_stores)
transport_costs = np.random.uniform(2, 15, (num_warehouses, num_stores))
max_warehouses = 5
max_units = 1500

x = m.addVars(I, vtype=GRB.BINARY, name="x")
z = m.addVars(I, J, vtype=GRB.CONTINUOUS, name="z")
y = m.addVars(I, J, vtype=GRB.BINARY, name="y")

for j in J:
    m.addConstr(gp.quicksum(y[i, j] for i in I) == 1)

m.addConstr(gp.quicksum(x[i] for i in I) <= max_warehouses)

for i in I:
    for j in J:
        m.addConstr(y[i, j] <= x[i])
        m.addConstr(z[i, j] == demands[j] * y[i, j])
    m.addConstr(gp.quicksum(z[i, j] for j in J) <= max_units * x[i])

m.setObjective(
    gp.quicksum(fixed_costs[i] * x[i] for i in I) + 
    gp.quicksum(transport_costs[i, j] * z[i, j] for i in I for j in J), 
    GRB.MINIMIZE
)
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"Total Minimized Cost: €{m.objVal:.2f}")
    print("Opened Warehouses:")
    for i in I:
        if x[i].x > 0.5:
            stores_served = [j for j in J if y[i,j].x > 0.5]
            print(f"- Warehouse {i} serving stores {stores_served} (Total units: {sum(z[i,j].x for j in stores_served):.0f})")


Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))
CPU model: 13th Gen Intel(R) Core(TM) i7-13620H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads
Optimize a model with 431 rows, 420 columns and 1240 nonzeros (Min)
Model fingerprint: 0x51fce251
Model has 220 linear objective coefficients
Variable types: 200 continuous, 220 integer (220 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [2e+00, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]
Presolve removed 200 rows and 200 columns
Presolve time: 0.00s
Presolved: 231 rows, 220 columns, 840 nonzeros
Variable types: 0 continuous, 220 integer (220 binary)
Found heuristic solution: objective 477753.52583
Found heuristic solution: objective 416644.67077
Found heuristic solution: objective 188861.68554
Found heuristic solution: objective 188835.95569
Root relaxation: objective 1.131175e+